# Experiment Tracking — MLflow (Phase 1 / MLOps)

Every model run in Pipelines 1 and 2 logs its parameters, per-fold-aggregated metrics (mean/std of R2, RMSE, NRMSE, MAE), the generalization gap, and the raw fold table as an artifact to a local SQLite-backed MLflow tracking store (`mlflow.db` at the repo root). This notebook queries that store directly (no retraining) and summarizes every experiment and run.

To browse runs interactively (with the plots/artifacts UI), launch:

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

All summary tables produced here are saved to `results/mlflow/`, and the last cell shows the complete combined results.

Setup: point MLflow at the local SQLite store and configure pandas to display full tables.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)

from src.config import PROJECT_ROOT, get_path
from src.utils.io import save_table

db_path = PROJECT_ROOT / 'mlflow.db'
mlflow.set_tracking_uri(f'sqlite:///{db_path.as_posix()}')
client = MlflowClient()
print('Tracking URI:', mlflow.get_tracking_uri())

## Experiments overview

Every experiment corresponds to one `pipeline{1,2}_<dataset>` run (Pipelines 1 and 2 log to MLflow; Pipelines 3/4, tuning and XAI do not).

In [ ]:
experiments = client.search_experiments()
overview = pd.DataFrame(
    [
        {
            'experiment_id': e.experiment_id,
            'experiment_name': e.name,
            'n_runs': len(client.search_runs([e.experiment_id])),
        }
        for e in experiments
    ]
).sort_values('experiment_name').reset_index(drop=True)
overview

## All runs, every experiment

`mlflow.search_runs` returns one row per run with every logged parameter and metric as a column.

In [ ]:
all_runs = mlflow.search_runs(experiment_ids=[e.experiment_id for e in experiments], search_all_experiments=True)
print('Total logged runs:', len(all_runs))
all_runs

## Best run per experiment (by mean test R2)

In [ ]:
if 'metrics.test_r2_mean' in all_runs.columns:
    best_per_experiment = (
        all_runs.sort_values('metrics.test_r2_mean', ascending=False)
        .groupby('experiment_id', as_index=False)
        .first()
    )
    key_cols = [
        c for c in best_per_experiment.columns
        if c in ('experiment_id', 'run_id', 'tags.mlflow.runName', 'status')
        or c.startswith('params.') or c.startswith('metrics.')
    ]
    best_per_experiment = best_per_experiment.merge(
        overview[['experiment_id', 'experiment_name']], on='experiment_id'
    )
    display_cols = ['experiment_name'] + [c for c in key_cols if c != 'experiment_id']
    best_per_experiment[display_cols]
else:
    print('No runs with metrics.test_r2_mean found yet — run notebook 02 first.')
    best_per_experiment = pd.DataFrame()
    display_cols = []

## Top 15 runs overall (by mean test R2)

In [ ]:
if not all_runs.empty and 'metrics.test_r2_mean' in all_runs.columns:
    metric_cols = [c for c in all_runs.columns if c.startswith('metrics.')]
    param_cols = [c for c in all_runs.columns if c.startswith('params.')]
    top_runs = all_runs.sort_values('metrics.test_r2_mean', ascending=False).head(15)
    top_runs[['experiment_id', 'tags.mlflow.runName'] + param_cols + metric_cols]
else:
    top_runs = pd.DataFrame()
    print('No runs available yet.')

## Final Summary — Complete MLflow Tracking Overview

Experiments overview, every run, and the best run per experiment, saved to `results/mlflow/`.

In [ ]:
mlflow_out_dir = get_path('results_dir') / 'mlflow'
save_table(overview, mlflow_out_dir / 'experiments_overview.csv')
save_table(all_runs, mlflow_out_dir / 'all_runs.csv')
if not best_per_experiment.empty:
    save_table(best_per_experiment, mlflow_out_dir / 'best_run_per_experiment.csv')

print('=' * 90)
print('1) EXPERIMENTS OVERVIEW')
print('=' * 90)
display(overview)

print()
print('=' * 90)
print(f'2) ALL {len(all_runs)} LOGGED RUNS')
print('=' * 90)
display(all_runs)

if not best_per_experiment.empty:
    print()
    print('=' * 90)
    print('3) BEST RUN PER EXPERIMENT (highest mean test R2)')
    print('=' * 90)
    display(best_per_experiment[display_cols])

print()
print('MLflow tracking store:', db_path.resolve())
print('Launch the UI with: mlflow ui --backend-store-uri sqlite:///mlflow.db')
print('Summary tables saved to:', mlflow_out_dir.resolve())